# 자연어 처리 1일차 (1) — 자연어 처리 과정 · 텍스트 전처리 · 형태소 · NLTK 토큰화 · OOV

수업 메모 `11_03메모` 를 **실행 가능한 코드로 확인하며 정리 · 개선한 노트**입니다.
(교재 `자연어처리` 교재9 p.1~14 — 자연어 · 자연어 처리 과정 · 텍스트 전처리 · 텍스트 구조 · 형태소 · 영어 / 한글 불용어 · 토큰화 · NLTK 실습 · 단어 단위 토큰화의 문제점)

- 서브워드 토큰화 **BPE** 는 [`bpe_subword_tokenization.ipynb`](./bpe_subword_tokenization.ipynb), **WordPiece** 단어 사전 만들기(필기 `단어사전`)는 [`wordpiece_vocab_nsmc.ipynb`](./wordpiece_vocab_nsmc.ipynb) 에 이어서 정리했습니다.
- 실습 데이터 : 네이버 영화 리뷰 **NSMC** (`ratings_train.txt` 15만 · `ratings_test.txt` 5만, 탭 구분 `id · document · label`)
- 메모에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

In [1]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 자연어 처리 과정 — 5단계 (교재 p.2~5)

| 단계 | 하는 일 | 오늘 실습 |
|---|---|---|
| ① 텍스트 데이터 수집 | 공공데이터 · 오픈 API · 웹 크롤링 | NSMC 영화 리뷰 20만 개 |
| ② 텍스트 전처리 | 특수문자 · **불용어** 제거 | 정규식 · NLTK 불용어 · 형태소 품사 |
| ③ 토큰화 | 의미 있는 단위로 나누기 | NLTK 문장 / 단어 · 형태소 · **서브워드** |
| ④ 임베딩 | 토큰 → 숫자 벡터 | 토큰 → **id** 까지 (벡터는 다음 시간) |
| ⑤ 모델 학습 | 목적 달성 (감성 분류 등) | — |

- **자연어** = 한글 · 영어처럼 사람이 쓰는 말 ↔ 인공 언어(프로그래밍 언어)
- **언어 모델(LM)** = 문맥을 보고 **다음 단어를 맞추는** 알고리즘 — LLM 의 출발점

![자연어 처리 5단계 애니메이션](./images/anim_01_nlp_pipeline.gif)

In [2]:
import os, zipfile, urllib.request

# ratings_train.txt(14.6MB)는 저장소에 ratings_train.zip 으로 올려 두었다 → 처음 한 번 압축 풀기
# 둘 다 없으면 원본 저장소(github.com/e9t/nsmc)에서 내려받기
for name in ['ratings_train.txt', 'ratings_test.txt']:
    if os.path.exists(name):
        continue
    zip_name = name.replace('.txt', '.zip')
    if os.path.exists(zip_name):
        zipfile.ZipFile(zip_name).extractall('.')
    else:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/e9t/nsmc/master/' + name, name)
print({n: f'{os.path.getsize(n) / 1e6:.1f}MB' for n in ['ratings_train.txt', 'ratings_test.txt']})

{'ratings_train.txt': '14.6MB', 'ratings_test.txt': '4.9MB'}


In [3]:
import re
import numpy as np
import pandas as pd

# ① 수집 : 네이버 영화 리뷰 (탭으로 구분된 텍스트 파일)
train = pd.read_csv('ratings_train.txt', sep='\t')
test = pd.read_csv('ratings_test.txt', sep='\t')
print('train :', train.shape, '| test :', test.shape)
print('리뷰가 비어 있는 행 :', train['document'].isna().sum(), '/', test['document'].isna().sum())
print('긍정(1) 비율 : %.3f' % train['label'].mean())
train.head()

train : (150000, 3) | test : (50000, 3)
리뷰가 비어 있는 행 : 5 / 3
긍정(1) 비율 : 0.499


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


## 2. 텍스트 데이터의 구조 (교재 p.7)

**말뭉치(corpus) ⊃ 문서(document) ⊃ 문단 ⊃ 문장(sentence) ⊃ 단어(word) ⊃ 형태소** — 큰 단위가 작은 단위를 품는 계층이다.

NSMC 로 보면 : 말뭉치 = 리뷰 20만 개 전체, 문서 = 리뷰 1개(`document` 열), 그 안에 문장 · 단어 · 형태소.

In [4]:
corpus = pd.concat([train['document'], test['document']]).dropna().tolist()
doc = '평점이 왜 이래요? 배우들 연기도 좋았고 스토리도 탄탄했다. 강추!'

sentences = re.split(r'(?<=[.!?])\s+', doc)        # 문장 : 끝 부호(. ! ?) 뒤에서 나누기
words = doc.split()                                 # 단어(어절) : 띄어쓰기로 나누기
print('말뭉치 :', f'{len(corpus):,} 개 문서')
print('문서   :', doc)
print('문장   :', sentences)
print('단어   :', words)

말뭉치 : 199,992 개 문서
문서   : 평점이 왜 이래요? 배우들 연기도 좋았고 스토리도 탄탄했다. 강추!
문장   : ['평점이 왜 이래요?', '배우들 연기도 좋았고 스토리도 탄탄했다.', '강추!']
단어   : ['평점이', '왜', '이래요?', '배우들', '연기도', '좋았고', '스토리도', '탄탄했다.', '강추!']


> **📝 고친 부분** (메모 `"말뭉치" 단계를 이룬다. (명어 단어)`)
> - 말뭉치는 '단계' 하나가 아니라 **가장 큰 단위**. 말뭉치 > 문서 > 문단 > 문장 > 단어 > 형태소 순서로 **계층**을 이룬다

## 3. 텍스트 전처리 — 특수문자 · 불용어 (교재 p.6)

- **특수문자 제거** : 의미 없는 기호 · 반복 문자 정리
- **불용어(stopword)** : **자주 나오지만** 분석에 큰 의미가 없는 단어 (영어 be 동사 · 관사 등)

In [5]:
samples = train['document'].dropna().iloc[[0, 3, 8, 26]].tolist()

def clean_text(text):
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', text)   # 한글 · 영문 · 숫자 · 공백만 남기기
    return re.sub(r'\s+', ' ', text).strip()                    # 여러 칸 공백 → 한 칸

for s in samples:
    print('원문 :', s)
    print('정리 :', clean_text(s))
    print()

원문 : 아 더빙.. 진짜 짜증나네요 목소리
정리 : 아 더빙 진짜 짜증나네요 목소리

원문 : 교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
정리 : 교도소 이야기구먼 솔직히 재미는 없다 평점 조정

원문 : 액션이 없는데도 재미 있는 몇안되는 영화
정리 : 액션이 없는데도 재미 있는 몇안되는 영화

원문 : kl2g 고추를 털어버려야 할텐데
정리 : kl2g 고추를 털어버려야 할텐데



In [6]:
import nltk
nltk.download('punkt_tab', quiet=True)        # 문장 부호 · 약어 규칙 (구버전 nltk 는 'punkt')
nltk.download('stopwords', quiet=True)        # 불용어 목록

stopwords_list = nltk.corpus.stopwords.words('english')     # 영어 불용어 리스트 생성
print('영어 불용어 수 :', len(stopwords_list), '(NLTK 데이터 버전에 따라 다르다 — 예전 목록은 179개)')
print(stopwords_list[:30])

영어 불용어 수 : 198 (NLTK 데이터 버전에 따라 다르다 — 예전 목록은 179개)
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't"]


In [7]:
from collections import Counter

# 불용어는 '드문' 단어가 아니라 '흔한' 단어다 — 영어 문장에서 빈도 상위 단어를 보면 대부분 불용어
text_en = ('The movie was good and the actors were great. The story was not boring, '
           'but the ending was too long and the music was a bit loud. I think it is worth watching.')
tokens_en = [w.lower() for w in nltk.word_tokenize(text_en) if w.isalpha()]
top = Counter(tokens_en).most_common(8)
print('빈도 상위 8 :', [(w, c, '불용어' if w in stopwords_list else '') for w, c in top])

no_stop = [w for w in tokens_en if w not in stopwords_list]
print()
print('불용어 제거 전 :', len(tokens_en), '개', tokens_en[:12])
print('불용어 제거 후 :', len(no_stop), '개', no_stop)
print("'not' 도 불용어 목록에 있음 →", 'not' in stopwords_list, "(감성 분석에서는 빼면 안 될 수도)")

빈도 상위 8 : [('the', 5, '불용어'), ('was', 4, '불용어'), ('and', 2, '불용어'), ('movie', 1, ''), ('good', 1, ''), ('actors', 1, ''), ('were', 1, '불용어'), ('great', 1, '')]

불용어 제거 전 : 33 개 ['the', 'movie', 'was', 'good', 'and', 'the', 'actors', 'were', 'great', 'the', 'story', 'was']
불용어 제거 후 : 14 개 ['movie', 'good', 'actors', 'great', 'story', 'boring', 'ending', 'long', 'music', 'bit', 'loud', 'think', 'worth', 'watching']
'not' 도 불용어 목록에 있음 → True (감성 분석에서는 빼면 안 될 수도)


> **📝 고친 부분** (메모 `불용어 제거: be 동사 나 잘 쓰이지 않는 문장을 순위를 맥여서 제거 한다.`)
> - 불용어는 **잘 안 쓰이는** 것이 아니라 **너무 자주 쓰여서** 구별에 도움이 안 되는 **단어**(문장이 아님). 빈도 순위를 매기면 위쪽에 몰려 있다
> - NLTK 영어 불용어에는 `not` · `no` 도 들어 있다 → 긍정 / 부정 분류에서는 목록을 그대로 쓰지 말고 **필요한 단어는 빼고** 써야 한다

## 4. 형태소 (교재 p.8~9)

**형태소** = 뜻을 가진 가장 작은 말의 단위(더 나누면 뜻이 사라짐). 혼자 쓰일 수 있는 말의 단위는 **단어** (단, 학교 문법에서는 혼자 못 쓰이는 **조사도 단어**로 본다 — `꽃이` 는 단어 2개).

| 분류 | 의미 | 예 |
|---|---|---|
| **실질 형태소** | 실제 뜻 (명사 · 동사 / 형용사 어간 · 부사) | 꽃밭 · 꽃 · 피- · 많이 · 사과 |
| **형식 형태소** | 문법적 관계만 (**조사 · 어미 · 접사**) | 에 · 이 · -었- · -다 · 풋- · -ness |

한글 형태소 분석기 `kiwipiepy` 로 교재 예시를 그대로 확인 (교재에는 없는 도구, `pip install kiwipiepy`).

In [8]:
# Colab : !pip install kiwipiepy
from kiwipiepy import Kiwi

kiwi = Kiwi()
TAG = {'NNG': '명사', 'NNP': '고유명사', 'JKS': '조사(주격)', 'JKB': '조사(부사격)', 'JKO': '조사(목적격)',
       'JX': '조사(보조사)', 'MAG': '부사', 'VV': '동사 어간', 'VA': '형용사 어간', 'VA-I': '형용사 어간',
       'EP': '어미(선어말)', 'EF': '어미(종결)', 'EC': '어미(연결)', 'ETM': '어미(관형형)',
       'XPN': '접두사', 'XSN': '명사 파생 접미사', 'SF': '문장부호'}

def morphs(text):
    return [(t.form, TAG.get(t.tag, t.tag)) for t in kiwi.tokenize(text)]

for text in ['꽃밭에 꽃이 많이 피었다', '책가방', '풋사과', '사과']:
    print(f'{text:12s} →', morphs(text))

꽃밭에 꽃이 많이 피었다 → [('꽃밭', '명사'), ('에', '조사(부사격)'), ('꽃', '명사'), ('이', '조사(주격)'), ('많이', '부사'), ('피', '동사 어간'), ('었', '어미(선어말)'), ('다', '어미(종결)')]
책가방          → [('책가방', '명사')]
풋사과          → [('풋', '접두사'), ('사과', '명사')]
사과           → [('사과', '명사')]


- `꽃밭에` = 꽃밭(명사) + 에(조사), `피었다` = 피(어간) + 었(어미) + 다(어미) — 교재 p.9 와 같다
- `사과` 는 더 나누면 뜻이 없어지는 **형태소 하나**(명사)이자 혼자 쓰이는 **단어**. `풋사과` 는 접두사 `풋-` + 명사 `사과`

> **📝 고친 부분**
> - 메모 `"사과" 단어를 단계가 없다 불용어` → `사과` 는 **더 나눌 수 없는 형태소 하나**(실질 형태소, 명사)이고 **불용어가 아니다**. 뜻이 있는 핵심 단어다
> - 메모 `형식형태소 찾는법: "접사,어미,조사" (불용어)를 3개를 찾으면 된다` → 형식 형태소는 **조사 · 어미 · 접사 세 종류**라는 뜻. 3개를 '찾는' 것이 아니고, 이 중 한글 **불용어로 지우는 것은 주로 조사 · 어미**(접사 `풋-` 을 지우면 뜻이 바뀐다)

## 5. 영어와 한글의 불용어 차이 (교재 p.9)

| | 불용어 단위 | 예 |
|---|---|---|
| 영어 | **단어** (대명사 · 관사 · 접속사 · 조동사 등) | the, a, and, is, it |
| 한글 | **형태소** (조사 · 어미 등) | 에, 이, 는, 었, 다 |

영어는 띄어쓰기 단위 단어를 통째로 지우면 되지만, 한글은 `꽃밭에` 처럼 한 어절 안에 조사가 붙어 있어 **형태소 분석 후** 지워야 한다.

![불용어 제거 애니메이션](./images/anim_05_stopwords_en_ko.gif)

In [9]:
def remove_ko_stopwords(text):
    # 품사 태그가 J(조사) · E(어미) · S(부호)로 시작하면 불용어로 보고 제거
    return [t.form for t in kiwi.tokenize(text) if not t.tag.startswith(('J', 'E', 'S'))]

for text in ['꽃밭에 꽃이 많이 피었다', '배우들 연기도 좋았고 스토리도 탄탄했다.']:
    print('띄어쓰기 :', text.split())
    print('불용어 제거 :', remove_ko_stopwords(text))
    print()

띄어쓰기 : ['꽃밭에', '꽃이', '많이', '피었다']
불용어 제거 : ['꽃밭', '꽃', '많이', '피']

띄어쓰기 : ['배우들', '연기도', '좋았고', '스토리도', '탄탄했다.']
불용어 제거 : ['배우', '들', '연기', '좋', '스토리', '탄탄하']



> **📝 고친 부분** (메모 `한글(불용어) 조사,어미,조사 등 형태소` · `영어(불용어) 단어(대명사,관사,관속사,조동사, 등)`)
> - 조사가 두 번 → **조사 · 어미** 등 형태소, `관속사` → **접속사**

## 6. 토큰화 — NLTK `sent_tokenize` · `word_tokenize` (교재 p.10~13)

- **토큰화** = 텍스트를 의미 있는 단위(토큰)로 나누기 — 결과 자료형은 **리스트**
- 문장 토큰화 : 끝 부호(. ! ?) 기준 + 약어(Mr. Dr.)는 문장 끝으로 보지 않음
- 단어 토큰화 : 띄어쓰기 기준 + **구두점 · 축약(n't)도 따로** 분리
- 영어 = 문장 / 띄어쓰기(단어) 기준, 한글 = 문장 / **형태소** 기준

In [10]:
from nltk.tokenize import sent_tokenize, word_tokenize

text1 = 'Hello Mr. Smith. How are you?'
print('split(\'. \')     :', text1.split('. '))           # 단순 마침표 분리 → Mr. 에서 잘못 잘림
print('sent_tokenize   :', sent_tokenize(text1))            # 약어 Mr. 를 인식

text2 = "Don't hesitate, just do it."
print()
print('split()         :', text2.split())                  # 쉼표 · 마침표가 단어에 붙어 있음
print('word_tokenize   :', word_tokenize(text2))            # Do / n't · 구두점 분리
print('자료형 :', type(word_tokenize(text2)))

split('. ')     : ['Hello Mr', 'Smith', 'How are you?']
sent_tokenize   : ['Hello Mr. Smith.', 'How are you?']

split()         : ["Don't", 'hesitate,', 'just', 'do', 'it.']
word_tokenize   : ['Do', "n't", 'hesitate', ',', 'just', 'do', 'it', '.']
자료형 : <class 'list'>


In [11]:
# 한글 리뷰에 같은 방법을 쓰면?
review = '평점이 왜 이래요? 배우들 연기도 좋았고 스토리도 탄탄했다. 강추!'
print('sent_tokenize :', sent_tokenize(review))       # 문장은 부호 기준이라 한글도 된다
print('word_tokenize :', word_tokenize(review))       # 어절 + 부호 — '평점이' 에 조사 '이' 가 붙은 채
print('형태소        :', [t.form for t in kiwi.tokenize(review)])

sent_tokenize : ['평점이 왜 이래요?', '배우들 연기도 좋았고 스토리도 탄탄했다.', '강추!']
word_tokenize : ['평점이', '왜', '이래요', '?', '배우들', '연기도', '좋았고', '스토리도', '탄탄했다', '.', '강추', '!']
형태소        : ['평점', '이', '왜', '이렇', '어요', '?', '배우', '들', '연기', '도', '좋', '었', '고', '스토리', '도', '탄탄하', '었', '다', '.', '강추', '!']


> **📝 고친 부분**
> - 메모 `NLIK` → **NLTK** (Natural Language Toolkit)
> - 메모 `단어 단위 마침표 기준으로 '인식'한다` → 마침표 기준은 **문장** 토큰화. 단어 토큰화는 **띄어쓰기 + 구두점** 기준
> - 메모 예시 두 개(`Hello Mr. Smith.` · `Don't hesitate`)는 실제 실행 결과와 **같다** ✔

## 7. 단어 단위 토큰화의 문제점 — OOV (교재 p.14)

**단어 사전(word → id)** 은 학습 데이터에서 만든다. 실제 데이터에 사전에 없는 단어가 나오면 **OOV(Out-Of-Vocabulary)** → 모두 `[UNK]` 하나로 뭉개진다. 신조어 · 오타 · 조사가 붙은 변형이 많은 한글 리뷰에서 특히 심하다.

In [12]:
train_docs = train['document'].dropna()
test_docs = test['document'].dropna()

# 띄어쓰기(어절) 단위 단어 사전 : 학습 데이터에서 많이 나온 30,000개
word_counts = Counter(w for d in train_docs for w in d.split())
vocab_word = {w for w, _ in word_counts.most_common(30000)}
print(f'학습 데이터의 서로 다른 어절 수 : {len(word_counts):,}  → 사전에 30,000개만')

test_words = [w for d in test_docs for w in d.split()]
oov = [w for w in test_words if w not in vocab_word]
print(f'테스트 어절 {len(test_words):,}개 중 OOV {len(oov):,}개 → {len(oov) / len(test_words):.1%}')
print('OOV 예 :', [w for w, _ in Counter(oov).most_common(15)])

학습 데이터의 서로 다른 어절 수 : 357,862  → 사전에 30,000개만
테스트 어절 380,472개 중 OOV 137,388개 → 36.1%


OOV 예 : ['하지마루요~유루유리', '밥그릇', '책임져라!!성우', '쓰게', '등록이', '타이틀', '있고...', '여기나오는', '지경.', '감동!!', '해주지', '첨봄', '개념', '로봇의', '1화']


In [13]:
# 같은 '재미' 라도 조사 · 어미가 붙으면 전부 다른 단어
variants = [w for w in word_counts if w.startswith('재미')]
print(f"'재미' 로 시작하는 서로 다른 어절 : {len(variants):,}개")
print(sorted(variants, key=lambda w: -word_counts[w])[:20])

for w in ['존잼', '갓띵작', '노잼각', '꿀잼보장']:
    print(f'{w:6s} 학습 빈도 {word_counts.get(w, 0):4d} → 사전에 있음? {w in vocab_word}')

'재미' 로 시작하는 서로 다른 어절 : 2,362개
['재미있게', '재미', '재미도', '재미가', '재미없다', '재미는', '재미있는', '재미없는', '재미없음', '재미있고', '재미없고', '재미없다.', '재미를', '재미있어요', '재미있다', '재미없어', '재미있다.', '재미와', '재미있음', '재미없게']
존잼     학습 빈도   13 → 사전에 있음? True
갓띵작    학습 빈도    0 → 사전에 있음? False
노잼각    학습 빈도    0 → 사전에 있음? False
꿀잼보장   학습 빈도    0 → 사전에 있음? False


- 어절 사전 3만 개로는 테스트 어절의 **36%(3개 중 1개 이상)** 가 OOV — 감성을 드러내는 신조어(`노잼각` 등)가 `[UNK]` 가 되면 분류가 틀어진다(교재 p.14 '가사 감정 분류' 예시와 같은 문제)
- 해결 : 단어를 더 작은 **서브워드** 조각으로 나눠 조합 → [`bpe_subword_tokenization.ipynb`](./bpe_subword_tokenization.ipynb) · [`wordpiece_vocab_nsmc.ipynb`](./wordpiece_vocab_nsmc.ipynb)

## 📝 고친 부분 정리 (메모 `11_03메모`)

| 메모 | 고친 것 |
|---|---|
| 불용어 = "잘 쓰이지 않는 문장" | **자주 쓰이지만** 의미가 적은 **단어** |
| "말뭉치" 단계를 이룬다 | 말뭉치 > 문서 > 문단 > 문장 > 단어 > 형태소 **계층** |
| "사과" … 불용어 | 더 나눌 수 없는 **형태소 하나(명사)**, 불용어 아님 |
| 형식형태소 "3개를 찾으면 된다" | 조사 · 어미 · 접사 **세 종류**, 한글 불용어는 주로 조사 · 어미 |
| 조사, 어미, 조사 / 관속사 | 조사 · 어미 / **접속사** |
| NLIK | **NLTK** |
| 단어 단위 = 마침표 기준 | 단어 = **띄어쓰기 + 구두점**, 마침표는 **문장** 기준 |